In [ ]:
import os

from typing import Any, Dict, List

import geopandas as gpd
import matplotlib.pyplot as plt
import shapely.geometry as sg

In [ ]:
from typing import Iterable, Optional


def yolo_string_to_dict(yolo_annotation: str) -> Dict[str, Any]:
    values = yolo_annotation.split()

    if len(values) >= 6:
        cat_id, x_center, y_center, width, height, conf = map(float, values[:6])
    elif len(values) == 5:
        cat_id, x_center, y_center, width, height = map(float, values)
        conf = 1.0
    else:
        raise ValueError(f"Unrecognized annotation string format: {yolo_annotation}")
    
    bbox = sg.box(
        minx=x_center-width/2,
        miny=y_center-height/2,
        maxx=x_center+width/2,
        maxy=y_center+height/2,
    )

    data = {
        "category": int(cat_id),
        "confidence": conf,
        "geometry": bbox
    }

    return data

def yolo_file_to_dicts(annotation_file_path: str) -> List[Dict[str, Any]]:
    data = []

    with open(annotation_file_path, 'r') as f:
        file_name = os.path.basename(annotation_file_path)
        for line in f.readlines():
            line_data = yolo_string_to_dict(line)
            line_data["file_name"] = file_name
            data.append(line_data)
    
    return data

def read_annotations_folder(folder_path: str, categories: Optional[Iterable[int]]) -> gpd.GeoDataFrame:
    data = []
    annotation_files = [
        file for file in os.listdir(folder_path) 
        if os.path.splitext(file)[1] == ".txt"
    ]

    for file in annotation_files:
        data.extend(
            yolo_file_to_dicts(os.path.join(folder_path, file))
        )
    
    gdf = gpd.GeoDataFrame(
        data=data,
        columns=["file_name", "category", "confidence", "geometry"]
    )

    if categories is not None:
        gdf = gdf[gdf["category"].isin(categories)]

    return gdf

## Load predictions and labels

In [ ]:
predictions_folder = "../datasets/experiments/zwerfafval/predict/yolo26m_1920_v1-2_extra_250-2/val/"
labels_folder = "../datasets/experiments/zwerfafval/dataset_v1-3_extra/labels/val/"

categories = {
    0: "zwerfafval_grof",
    1: "zwerfafval_fijn"
}

In [ ]:
labels_gdf = read_annotations_folder(labels_folder, categories.keys())
pred_gdf = read_annotations_folder(predictions_folder, categories.keys())

In [ ]:
# Plot ground truth bounding boxes to see their distribution in the image

fig, ax = plt.subplots(figsize=(10, 10))

labels_gdf.plot(column="category", categorical=True, alpha=0.5, legend=True, ax=ax)

ax.yaxis.set_inverted(True)

In [ ]:
# Plot prediction bounding boxes to see their distribution in the image

fig, ax = plt.subplots(figsize=(10, 10))

pred_gdf.plot(column="category", categorical=True, alpha=0.5, legend=True, ax=ax)

ax.yaxis.set_inverted(True)

In [ ]:
# Plot predictions and ground truth together for one image

file_name = "frame_10564.txt"

fig, ax = plt.subplots(figsize=(10, 10))

labels_gdf[labels_gdf["file_name"] == file_name].plot(alpha=0.5, color="green", ax=ax)
pred_gdf[pred_gdf["file_name"] == file_name].plot(alpha=0.5, color="red", ax=ax)

ax.yaxis.set_inverted(True)

## Compute statistics

In [ ]:
# We measure recall based on whether the centroid of a ground truth bounding box is covered by a prediction bounding box
# We measure precision based on whether a prediction bounding box includes the centroid of a ground truth bounding box

recall_gdf = labels_gdf.set_geometry(labels_gdf.centroid).sjoin(pred_gdf, how="left", predicate="covered_by", on_attribute="file_name")
precision_gdf = pred_gdf.sjoin(labels_gdf.set_geometry(labels_gdf.centroid), how="left", predicate="contains", on_attribute="file_name")

In [ ]:
recall_gdf[recall_gdf["file_name"]==file_name]

In [ ]:
precision_gdf[precision_gdf["file_name"]==file_name]

**TODO:**
- Compute precision and recall over confidence score
- Plot object counts GT versus PRED over the "route" (sorted images)
- Distinction between stats per category versus category agnostic (garbage is garbage)
- Compare TRAIN and VAL
- Compare different model sizes / input sizes
